# Explicabilidad

## Imports

In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, learning_curve
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import joblib

import matplotlib.pyplot as plt

import os
import sys
import numpy as np

from dotenv import load_dotenv

# Cargar las del archivo .env (si existe)
load_dotenv()

True

In [2]:
# Custom modules
# Obtener el directorio del notebook (notebooks/)
notebook_dir = os.getcwd()
# Subir un nivel para llegar a la raíz del proyecto
project_root = os.path.abspath(os.path.join(notebook_dir, '..'))
# Agregar al path si no está (solo afecta a la instancia actual)
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from src.data_management import get_categorical_number_columns
from src.model_trainer import ModelTrainer

/Users/josep.esparrell/Code/MDS/salary-anomaly-detection/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Cargar Dataset

In [3]:
file_evaluation = os.path.join('..','data', 'clean',os.getenv('FILE_EVALUACION'))
df = pd.read_csv(file_evaluation)

## Separacion de variables

In [4]:
# Variable objetivo (target)
y = df["MonthlyIncome"]

# Variables explicativas
X = df.drop(columns=["MonthlyIncome"])

print("\nShape de X:", X.shape)
print("Shape de y:", y.shape)


Shape de X: (294, 14)
Shape de y: (294,)


## Definir variables Categoricas y numericas

In [5]:
# Convertir columnas numéricas categóricas a tipo 'object'
categoricas_numericas = get_categorical_number_columns()

for col in categoricas_numericas:
    if col in X.columns:
        X[col] = X[col].astype("object")


# Obtener listas de variables categóricas y numéricas
cat_cols = X.select_dtypes(include=["object"]).columns.tolist()
num_cols = X.select_dtypes(exclude=["object"]).columns.tolist()

print("\nVariables categóricas:")
print(cat_cols)

print("\nVariables numéricas:")
print(num_cols)


Variables categóricas:
['Attrition', 'BusinessTravel', 'Department', 'Education', 'EducationField', 'JobInvolvement', 'JobRole', 'JobSatisfaction', 'OverTime', 'PerformanceRating', 'WorkLifeBalance']

Variables numéricas:
['JobLevel', 'YearsAtCompany', 'YearsSinceLastPromotion']


## Explicavilidad de las variables

In [6]:
# Cargar modelo ya entrenado

modelo_entrenado = joblib.load(os.path.join('..','models','mejor_modelo_salarial.joblib'))

# Extraer el preprocesador del pipeline guardado
preprocessor = modelo_entrenado.named_steps['preprocessor']

# Crear diccionario con el modelo cargado
modelos = {
    'RandomForest': modelo_entrenado.named_steps['model']
}

# Inicializar trainer con el modelo ya entrenado
trainer = ModelTrainer(
    modelos=modelos,
    preprocessor=preprocessor,
    figures_dir=os.path.join('..','outputs','figures'),
    models_dir=os.path.join('..','models')
)

# Registrar el pipeline completo en trainer (simular que ya fue entrenado)
trainer.pipelines['RandomForest'] = modelo_entrenado
trainer.mejor_modelo = modelo_entrenado
trainer.mejor_nombre = 'RandomForest'

# Ahora sí, usar métodos de explicabilidad con datos de evaluación
trainer.analizar_feature_importance('RandomForest', top_n=15)

# Muestra para SHAP e impacto de negocio
X_eval_sample = X.sample(n=min(500, len(X)), random_state=42)
y_eval_sample = y.loc[X_eval_sample.index]

trainer.analizar_impacto_negocio('RandomForest', X_eval_sample, y_eval_sample, top_n=10)
trainer.analizar_shap('RandomForest', X_eval_sample.sample(min(300, len(X_eval_sample)), random_state=42), max_display=15)


Feature Importance: RandomForest

📊 Top 15 Features más importantes:

                          feature  importance
                         JobLevel    0.928321
        JobRole_Research Director    0.010524
                   YearsAtCompany    0.008894
                  JobRole_Manager    0.005890
          YearsSinceLastPromotion    0.004291
    JobRole_Laboratory Technician    0.004008
          JobRole_Sales Executive    0.002367
JobRole_Healthcare Representative    0.001930
                JobSatisfaction_2    0.001578
       JobRole_Research Scientist    0.001550
                      Education_4    0.001462
   JobRole_Manufacturing Director    0.001404
                      Education_3    0.001399
     EducationField_Life Sciences    0.001359
                 JobInvolvement_3    0.001254

✓ Gráfico guardado: ../outputs/figures/feature_importance_RandomForest.png
✓ Datos guardados: ../outputs/figures/feature_importance_RandomForest.csv

Análisis de Impacto de Negocio: RandomFore

(array([[-5.36416826e+00, -1.98771425e+01,  4.71285592e+00, ...,
         -3.14556277e+03,  1.40342762e+02, -9.03172863e+00],
        [ 4.49776409e+00,  3.89429647e+00, -6.79221444e+01, ...,
         -1.16274153e+03,  2.62070745e+02,  2.47518969e+01],
        [ 3.81898811e+00,  3.08004817e+00,  7.22163233e+00, ...,
          9.07714095e+03,  1.05097608e+02, -2.15404745e+01],
        ...,
        [ 6.79158941e+00,  6.67138395e+00,  6.66053253e+00, ...,
          3.64915694e+03,  4.56961906e+01, -2.08349210e+00],
        [ 4.79621999e+00,  6.16603788e+00,  7.34299044e+00, ...,
          3.72172681e+03,  3.47113914e+02, -3.88752992e+01],
        [ 2.33248876e+01,  1.56773462e+01,  3.58563920e+00, ...,
         -3.05322434e+03,  7.49668881e+01, -3.77587158e+01]],
       shape=(294, 47)),
                               feature  shap_importance
 44                           JobLevel      3275.803496
 25      JobRole_Laboratory Technician       157.030544
 28          JobRole_Research Directo